# debim Ultra-Stress Testing Runner on Kaggle (Round 2: MEP & Proxies Enabled)

This notebook executes an automated, high-throughput roundtrip benchmark of **debim** (Declarative BIM)
against hundreds of real-world IFC building models (Architecture, Structure, MEP, HVAC, and generic proxies).

### Key Capabilities:
- **Process-Level Crash Isolation**: Corrupted models do not crash the batch.
- **45-second Hard Timeout**: Prevents hanging on ultra-dense meshes.
- **Full MEP & Distribution Support**: Evaluates ducts, pipes, fittings, terminals, proxies, and chimneys.
- **Fidelity Verification**: Measures Retention Rate (%) and LLM Token Reduction (%).

In [ ]:
# 1. Install dependencies and pull fresh latest debim main branch
!pip install -q ifcopenshell "pydantic>=2.0" pyyaml rich pandas tabulate psutil

!rm -rf /kaggle/working/debim
!git clone https://github.com/PRIDA-TAKON/debim.git /kaggle/working/debim

import sys
sys.path.insert(0, "/kaggle/working/debim/src")
sys.path.insert(0, "/kaggle/working/debim")
print("Environment initialized with latest debim commit.")

In [ ]:
# 2. Scan and discover all IFC files across all mounted /kaggle/input datasets
from pathlib import Path
from tools.kaggle.debim_kaggle_stress_test import scan_for_ifc_files

input_paths = [Path("/kaggle/input")]
ifc_files = scan_for_ifc_files(input_paths)
print(f"Found {len(ifc_files)} IFC files across all mounted datasets.")
for f in ifc_files[:8]:
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 3. Execute Parallel Stress Test with 4 Cores & 45s Hard Timeout
from tools.kaggle.debim_kaggle_stress_test import run_stress_test

output_dir = Path("/kaggle/working/output_round2")
run_stress_test(
    input_dirs=input_paths,
    output_dir=output_dir,
    max_files=None,  # Process all discovered models
    timeout_sec=45,
    workers=4,
    resume=True
)

In [ ]:
# 4. Display KPI Summary and Final Metrics
import sqlite3
import pandas as pd

db_path = output_dir / "debim_stress_test.sqlite"
conn = sqlite3.connect(str(db_path))
df = pd.read_sql_query("SELECT * FROM results", conn)

print("=== ROUND 2 KPI SUMMARY ===")
print(f"Total Processed: {len(df)}")
success_df = df[df["status"].isin(["SUCCESS", "PARTIAL_RETENTION"])]
print(f"Crash-Resilience Rate: {(len(success_df)/len(df)*100):.1f}%")
print(f"Average Retention: {success_df['retention_rate_pct'].mean():.1f}%")
print(f"Median Retention: {success_df['retention_rate_pct'].median():.1f}%")
print(f"Average Compression: {success_df['compression_ratio_pct'].mean():.1f}%")
tokens_saved = (success_df['estimated_tokens_ifc'] - success_df['estimated_tokens_yaml']).sum()
print(f"Total LLM Tokens Saved: {tokens_saved:,}")

display(df.head(30))